# 🚩 Sprint 1 · Checkpoint 01 — The Engine

**Companion ao Boyd (Convex Optimization, Cap 2.1) + Weinberger (Lec 1, 2, 12 parte 1).**

Este checkpoint constrói os tijolos: vetor, norma, produto interno, ângulo. Depois usa-os para o teu **primeiro algoritmo de otimização** — Gradient Descent em 1D.

**Como trabalhar:**
- Cada `# TODO` é uma micro-task. Substitui o `...` ou completa a expressão.
- A célula corre uma `assert` — se falhar, lê a mensagem e corrige.
- **NÃO edites este ficheiro** (`scaffolded.ipynb`). Trabalha em `solutions/local.ipynb` (cópia local).

**Regra de ouro:** se um exercício parece trivial, escreve à mão o que estás a calcular antes de codar. A intuição geométrica é o ponto, não o código.

## ⚙️ Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Reprodutibilidade — fixa a seed quando estiveres a debugar.
rng = np.random.default_rng(seed=42)

# Atalho para asserção com mensagem visual.
def check(cond, msg='falhou'):
    assert cond, f'❌ {msg}'
    print('✅', msg)

## §1 — Vetores: a base geométrica

**Aulas Weinberger:** L1 (Supervised Learning), L2 (KNN & Curse of Dimensionality).

Antes de minimizar funções, precisas das ferramentas para *medir* coisas: tamanho (norma), direção (vetor unitário), e relação angular (produto interno). Toda a regressão, classificação e otimização que farás depois usa estas três operações como blocos lego.

**Boyd 2.1 — pergunta para responderes em fim do checkpoint:** o que distingue um conjunto **afim** (ex: reta infinita) de um conjunto **convexo** (ex: segmento)?

### Task 1.1 — Sigmoid

A "porta lógica" suave que aparece em redes neurais e regressão logística:

$$ \sigma(x) = \frac{1}{1 + e^{-x}} $$

Propriedades que vais usar repetidamente:
- $\sigma(0) = 0.5$
- $\sigma(x) \to 1$ quando $x \to \infty$, $\sigma(x) \to 0$ quando $x \to -\infty$
- $\sigma(-x) = 1 - \sigma(x)$ (simetria)
- $\sigma'(x) = \sigma(x)(1 - \sigma(x))$ (gradiente compacto — útil para backprop)

Implementa de forma vectorizada (aceita arrays NumPy).

In [ ]:
def sigmoid(x):
    """σ(x) = 1 / (1 + e^{-x}). Aceita escalar ou array."""
    # TODO: usa np.exp e devolve a fórmula
    return ...

# Casos limite
check(abs(sigmoid(0.0) - 0.5) < 1e-9, 'sigmoid(0) deve ser exatamente 0.5')
check(sigmoid(100.0) > 0.999, 'sigmoid(+grande) deve aproximar 1')
check(sigmoid(-100.0) < 0.001, 'sigmoid(-grande) deve aproximar 0')

# Vectorização
xs = np.array([-2.0, -1.0, 0.0, 1.0, 2.0])
ys = sigmoid(xs)
check(ys.shape == xs.shape, f'shape esperado {xs.shape}, obteve {ys.shape}')

# Simetria σ(-x) = 1 - σ(x)
check(np.allclose(sigmoid(-xs), 1 - sigmoid(xs)),
      'σ(-x) deve ser igual a 1 - σ(x) (simetria)')

# Derivada: σ'(x) = σ(x)(1-σ(x)). Verifica numericamente em x=0.
h = 1e-5
deriv_num = (sigmoid(0 + h) - sigmoid(0 - h)) / (2 * h)
deriv_analy = sigmoid(0.0) * (1 - sigmoid(0.0))
check(abs(deriv_num - deriv_analy) < 1e-6,
      f'σ\'(0) numérica ≈ analítica? num={deriv_num:.6f}, analy={deriv_analy:.6f}')

### Task 1.2 — Produto interno (dot product)

$$ a \cdot b = \sum_i a_i b_i $$

Operação fundamental: mede **alinhamento**. Se for zero, vetores são ortogonais (90°). Se for positivo, apontam para o mesmo lado.

Implementa em duas versões:
1. **Manual** (loop) — para sentires a mecânica.
2. **NumPy** (`np.dot` ou `@`) — para performance.

Ambas devem dar o mesmo resultado.

In [ ]:
def dot_manual(a, b):
    """Produto interno via loop explícito."""
    a = np.asarray(a, dtype=float); b = np.asarray(b, dtype=float)
    assert a.shape == b.shape, 'vetores devem ter o mesmo shape'
    total = 0.0
    # TODO: itera com for e acumula a[i] * b[i] em total
    return total

def dot_numpy(a, b):
    """Produto interno via NumPy."""
    # TODO: usa np.dot ou o operador @
    return ...

a = np.array([1.0, 2.0, 3.0])
b = np.array([4.0, 5.0, 6.0])
check(abs(dot_manual(a, b) - 32.0) < 1e-9, f'1·4 + 2·5 + 3·6 = 32, obteve {dot_manual(a, b)}')
check(abs(dot_numpy(a, b) - dot_manual(a, b)) < 1e-9,
      'manual e numpy devem coincidir')

# Ortogonalidade
e1 = np.array([1.0, 0.0, 0.0])
e2 = np.array([0.0, 1.0, 0.0])
check(abs(dot_numpy(e1, e2)) < 1e-9, 'eixos canónicos são ortogonais → dot = 0')

# Auto-produto = norma ao quadrado
v = rng.normal(size=10)
check(abs(dot_numpy(v, v) - np.sum(v**2)) < 1e-9,
      'v · v deve ser igual a Σ v_i²')

### Task 1.3 — Norma Euclideana (L2)

Magnitude do vetor:

$$ \|v\|_2 = \sqrt{\sum_i v_i^2} = \sqrt{v \cdot v} $$

Repara: a norma é a raiz do auto-produto interno. Esta é a *primeira* ponte entre álgebra e geometria.

Implementa sem usar `np.linalg.norm` — força-te a ver a fórmula.

In [ ]:
def norm_l2(v):
    """Norma Euclideana (L2) sem np.linalg.norm."""
    v = np.asarray(v, dtype=float)
    # TODO: usa np.sum e np.sqrt para devolver sqrt(Σ v_i²)
    return ...

# Casos canónicos
check(abs(norm_l2([3.0, 4.0]) - 5.0) < 1e-9, '3-4-5 triângulo: ||(3,4)|| = 5')
check(abs(norm_l2([1.0, 0.0, 0.0]) - 1.0) < 1e-9, 'vetor unitário tem norma 1')
check(abs(norm_l2([0.0, 0.0, 0.0])) < 1e-9, 'vetor zero tem norma 0')

# Coerência com np.linalg.norm em vetor aleatório
v = rng.normal(size=7)
check(abs(norm_l2(v) - np.linalg.norm(v)) < 1e-9,
      'norma própria deve coincidir com np.linalg.norm')

### Task 1.4 — Normalizar um vetor

Transforma `v` num vetor unitário mantendo a direção:

$$ \hat{v} = \frac{v}{\|v\|} $$

Cuidado com o caso `v = 0` — divisão por zero. Em código de produção, ou levantas erro ou devolves o próprio zero. Aqui devolve `v` inalterado se a norma for `< 1e-12`.

In [ ]:
def normalize(v):
    """Devolve v / ||v||, ou v se ||v|| ≈ 0."""
    v = np.asarray(v, dtype=float)
    n = norm_l2(v)
    if n < 1e-12:
        return v
    # TODO: devolve v / n
    return ...

u = normalize([3.0, 4.0])
check(abs(norm_l2(u) - 1.0) < 1e-9, f'norma após normalizar deve ser 1, obteve {norm_l2(u):.6f}')
check(np.allclose(u, [0.6, 0.8]), f'(3,4)/5 = (0.6, 0.8), obteve {u}')

# Direção preserva-se
v = rng.normal(size=5)
u = normalize(v)
# u é múltiplo positivo de v: o cosseno do ângulo deve ser ~1
check(abs(np.dot(u, v) / norm_l2(v) - 1.0) < 1e-9,
      'após normalizar, direção mantém-se (cosseno com original = 1)')

# Caso degenerado
zero = np.array([0.0, 0.0, 0.0])
check(np.allclose(normalize(zero), zero), 'vetor zero deve ser devolvido inalterado')

### Task 1.5 — Ângulo entre vetores (cosine similarity)

$$ \cos\theta = \frac{a \cdot b}{\|a\| \|b\|} $$

Esta é literalmente a base da *semantic search*, dos *embeddings* de LLMs e da maior parte das medidas de similaridade em NLP.

Devolve o ângulo em **graus** (mais legível). `np.arccos` devolve radianos; converte com `np.degrees`.

In [ ]:
def angle_degrees(a, b):
    """Ângulo entre a e b, em graus, em [0, 180]."""
    a = np.asarray(a, dtype=float); b = np.asarray(b, dtype=float)
    cos_theta = dot_numpy(a, b) / (norm_l2(a) * norm_l2(b))
    # Clip para evitar erros numéricos (cos_theta sair ligeiramente fora de [-1, 1])
    cos_theta = np.clip(cos_theta, -1.0, 1.0)
    # TODO: usa np.arccos e np.degrees para converter para graus
    return ...

# Eixos canónicos: 90°
check(abs(angle_degrees([1, 0], [0, 1]) - 90.0) < 1e-6, 'eixos perpendiculares → 90°')
# Mesma direção: 0°
check(abs(angle_degrees([1, 2, 3], [2, 4, 6]) - 0.0) < 1e-6, 'múltiplos positivos → 0°')
# Direção oposta: 180°
check(abs(angle_degrees([1, 0], [-1, 0]) - 180.0) < 1e-6, 'direções opostas → 180°')
# Caso 45°
check(abs(angle_degrees([1, 0], [1, 1]) - 45.0) < 1e-6, '(1,0) vs (1,1) → 45°')

## §2 — Gradient Descent: o teu primeiro otimizador

**Aula Weinberger:** L12 parte 1 (Gradient Descent vs Newton — diferença geométrica).

A pergunta central da otimização: dado $f: \mathbb{R}^n \to \mathbb{R}$, qual o $x^*$ que minimiza $f(x)$?

A receita mais simples (a que vais codar agora):
1. Calcula $\nabla f(x)$ (a direção de subida mais íngreme).
2. Dá um passo no sentido **oposto**: $x_{\text{novo}} = x_{\text{velho}} - \eta \nabla f(x_{\text{velho}})$.
3. Repete até convergir.

O parâmetro $\eta$ é o *learning rate*. Pequeno demais → demora. Grande demais → diverge.

### Task 2.1 — GD em 1D, função convexa

Minimiza $f(x) = x^2 - 4x + 1$.

Derivada: $f'(x) = 2x - 4$. Mínimo analítico: $f'(x) = 0 \Rightarrow x^* = 2$.

Implementa `gd_1d(x0, lr, n_iter)` que devolve a **história** dos `x` (incluindo o inicial). Verás `x` a convergir para `2.0`.

In [ ]:
def f(x):
    return x**2 - 4*x + 1

def grad_f(x):
    """Derivada de f(x) = x² - 4x + 1."""
    # TODO: devolve 2x - 4
    return ...

def gd_1d(x0, lr, n_iter):
    """Devolve uma lista com [x0, x1, ..., x_{n_iter}]."""
    history = [float(x0)]
    x = float(x0)
    for _ in range(n_iter):
        # TODO: x = x - lr * grad_f(x)
        ...
        history.append(x)
    return history

# Sanity: gradiente em x=2 é zero
check(abs(grad_f(2.0)) < 1e-9, 'no mínimo (x=2), gradiente deve ser 0')
# Sanity: gradiente em x=0 é -4 (negativo → estamos à esquerda do mínimo)
check(abs(grad_f(0.0) - (-4.0)) < 1e-9, 'gradiente em 0 deve ser -4')

# Convergência com lr razoável
hist = gd_1d(x0=-5.0, lr=0.1, n_iter=50)
check(abs(hist[-1] - 2.0) < 1e-3,
      f'após 50 iterações deve estar próximo de x*=2, obteve {hist[-1]:.6f}')
# Cada passo deve aproximar do mínimo
errors = [abs(x - 2.0) for x in hist]
check(all(errors[i+1] <= errors[i] + 1e-9 for i in range(len(errors)-1)),
      'erro |x - 2| deve ser monotonamente decrescente com lr=0.1')

### Task 2.2 — Visualizar a trajetória

Plot de duas coisas:
1. `x` ao longo das iterações (deve aproximar de 2).
2. `f(x)` ao longo das iterações (deve aproximar do mínimo $f(2) = -3$).

Usa `gd_1d(x0=-5, lr=0.1, n_iter=30)`.

In [ ]:
hist = gd_1d(x0=-5.0, lr=0.1, n_iter=30)
fs = [f(x) for x in hist]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(hist, 'o-')
axes[0].axhline(2.0, color='r', linestyle='--', label='mínimo x*=2')
axes[0].set_xlabel('iteração'); axes[0].set_ylabel('x'); axes[0].legend()
axes[0].set_title('Trajetória de x')

axes[1].plot(fs, 'o-')
axes[1].axhline(f(2.0), color='r', linestyle='--', label='mínimo f*=-3')
axes[1].set_xlabel('iteração'); axes[1].set_ylabel('f(x)'); axes[1].legend()
axes[1].set_title('Trajetória de f(x)')
plt.tight_layout()
plt.show()

check(fs[-1] < fs[0], 'f(x) final deve ser menor que f(x) inicial')
check(fs[-1] - f(2.0) < 0.01, f'f(x) final deve estar perto do mínimo -3, obteve {fs[-1]:.4f}')

### Task 2.3 — Sensibilidade ao learning rate

Repete `gd_1d(x0=-5, n_iter=30)` para `lr ∈ [0.01, 0.1, 0.5, 0.99, 1.01]` e plota cada trajetória de `x` no mesmo gráfico.

**O que vais observar:**
- `lr=0.01`: lento — não chega ao mínimo em 30 passos.
- `lr=0.1`: convergência saudável.
- `lr=0.5`: mais rápido mas com overshoot.
- `lr=0.99`: oscila perto do mínimo, demora a estabilizar.
- `lr=1.01`: **diverge** (vai para infinito).

**Conexão Boyd Cap 9:** existe uma `lr` *ótima* para funções quadráticas — função do maior valor próprio do Hessiano. Vais derivar isto no Checkpoint 04.

In [ ]:
lrs = [0.01, 0.1, 0.5, 0.99, 1.01]
fig, ax = plt.subplots(figsize=(7, 5))
for lr in lrs:
    hist = gd_1d(x0=-5.0, lr=lr, n_iter=30)
    # Clip para o plot ser legível mesmo quando diverge
    hist = np.clip(hist, -1e3, 1e3)
    ax.plot(hist, 'o-', label=f'lr={lr}', markersize=3)
ax.axhline(2.0, color='k', linestyle='--', alpha=0.5, label='x*=2')
ax.set_xlabel('iteração'); ax.set_ylabel('x')
ax.set_title('GD 1D: sensibilidade ao learning rate')
ax.legend(); ax.set_ylim(-10, 15)
plt.show()

# Verifica divergência com lr=1.01
hist_div = gd_1d(x0=-5.0, lr=1.01, n_iter=30)
check(abs(hist_div[-1]) > abs(hist_div[0]),
      f'lr=1.01 deve divergir (|x| cresce), obteve hist[0]={hist_div[0]}, hist[-1]={hist_div[-1]:.2f}')

# Verifica convergência rápida com lr=0.5
hist_fast = gd_1d(x0=-5.0, lr=0.5, n_iter=30)
check(abs(hist_fast[-1] - 2.0) < 1e-6,
      'lr=0.5 deve convergir muito rápido neste problema (1-passo na verdade)')

## 🏁 Engine — fechado

Se chegaste aqui sem `AssertionError`, dominaste:

1. **Vetores** — sigmoid, dot product, norma, normalização, ângulo. A álgebra-base de tudo o que vem.
2. **Gradient Descent** — o algoritmo mais simples de otimização contínua, em 1D, com sensibilidade ao learning rate visualizada.

**Pergunta de fecho** (responde sem ver as respostas):
- O que distingue um conjunto **afim** de um conjunto **convexo**? (Boyd 2.1)
- Por que é que existe um `lr` máximo acima do qual GD diverge — o que é que esse limite tem a ver com a curvatura da função?
- Qual a diferença geométrica entre seguir o gradiente (GD) e seguir o passo de Newton (curvatura)?

**Próximo checkpoint:** [02 — Geometry](../02_Checkpoint_Geometry/) — sai do 1D, entra em matrizes e regressão multivariável.